# 05 — Results Comparison

This notebook loads the full experiment results from `metrics.jsonl` and produces comparison tables and visualisations.  
Each row in the JSONL file contains per-fold, per-seed metrics for one model on one event class.

## 1. Load Results

The `metrics.jsonl` file is produced by `run_experiment.py` and contains columns: model, seed, fold, event_class, event_name, n_params, threshold_percentile, threshold_value, n_train, n_test_normal, n_test_anomaly, train_time_s, f1, auc_roc, auc_pr, precision, recall.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load experiment results
df = pd.read_json("../../data/samarone_junior/results/metrics.jsonl", lines=True)
print(f"Results shape: {df.shape}")
print(f"Models: {df['model'].unique().tolist()}")
print(f"Event classes: {sorted(df['event_class'].unique().tolist())}")
df.head()

## 2. Aggregate Metrics by Model

We compute mean and standard deviation of key metrics (F1, AUC-ROC, AUC-PR) across all folds, seeds, and event classes.

In [ ]:
# Overall summary by model
summary = df.groupby("model").agg(
    f1_mean=("f1", "mean"),
    f1_std=("f1", "std"),
    auc_roc_mean=("auc_roc", lambda x: np.nanmean(x)),
    auc_roc_std=("auc_roc", lambda x: np.nanstd(x)),
    auc_pr_mean=("auc_pr", lambda x: np.nanmean(x)),
    auc_pr_std=("auc_pr", lambda x: np.nanstd(x)),
    n_params=("n_params", "first"),
    train_time_mean=("train_time_s", "mean"),
).round(4)

summary = summary.sort_values("f1_mean", ascending=False)
print(f"Full cross-validation: {len(df)} rows = {df['model'].nunique()} models × "
      f"{df['event_class'].nunique()} events × {df['seed'].nunique()} seeds × "
      f"{df['fold'].nunique()} folds")
summary

## 3. F1 Score Boxplots by Model

Boxplots show the distribution of F1 scores across folds and seeds, giving a sense of each model's consistency.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
model_order = summary.index.tolist()
sns.boxplot(data=df, x="model", y="f1", order=model_order, ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right", fontsize=9)
ax.set_ylabel("F1 Score")
ax.set_title("F1 Score Distribution by Model")
plt.tight_layout()
plt.show()

## 4. Event-Type Heatmap

A heatmap of mean F1 by event class (rows) and model (columns) reveals which fault types are easier or harder to detect.

In [ ]:
from qml.samarone_junior.loaders import ThreeWLoader

# Pivot table: event_class × model → mean F1
pivot = df.pivot_table(
    index="event_class", columns="model", values="f1", aggfunc="mean"
)
# Replace numeric indices with event names
pivot.index = [ThreeWLoader.EVENT_NAMES.get(i, f"Class {i}") for i in pivot.index]

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="YlOrRd", ax=ax, vmin=0, vmax=1)
ax.set_title("Mean F1 by Event Class and Model")
ax.set_ylabel("Event Class")
plt.tight_layout()
plt.show()

## 5. Per-Model Detailed Metrics

We show precision, recall, and F1 per model and event class for deeper analysis.

In [ ]:
# Detailed breakdown
detail = df.groupby(["model", "event_class"]).agg(
    f1_mean=("f1", "mean"),
    precision_mean=("precision", "mean"),
    recall_mean=("recall", "mean"),
    n_runs=("f1", "count"),
).round(3)
detail

## 6. LaTeX-Ready Summary Table

A formatted table suitable for the report, showing mean ± std for each metric.

In [ ]:
# LaTeX-ready table
latex_df = summary.copy()
latex_df["F1"] = latex_df.apply(lambda r: f"{r['f1_mean']:.3f} ± {r['f1_std']:.3f}", axis=1)
latex_df["AUC-ROC"] = latex_df.apply(lambda r: f"{r['auc_roc_mean']:.3f} ± {r['auc_roc_std']:.3f}", axis=1)
latex_df["AUC-PR"] = latex_df.apply(lambda r: f"{r['auc_pr_mean']:.3f} ± {r['auc_pr_std']:.3f}", axis=1)
latex_df["Params"] = latex_df["n_params"].fillna(0).astype(int).replace(0, "n/a")
latex_df["Train (s)"] = latex_df["train_time_mean"].apply(lambda x: f"{x:.1f}")
print(latex_df[["F1", "AUC-ROC", "AUC-PR", "Params", "Train (s)"]].to_latex())

## 7. Summary

- **3150 result rows**: 7 models × 9 event classes × 10 seeds × 5 folds.
- Boxplots reveal model consistency; the heatmap reveals per-event-class strengths.
- The LaTeX table above can be copied directly into the report.
- For publication-quality PDF figures, see notebook 06 which uses the dedicated `visualization.figures` module.